In [ ]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 13.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import string

# Load dataset
# Pastikan nama file sesuai dengan yang ada di folder Files
df = pd.read_csv('dataset_mentah_iphone17.csv')

# Kita hanya butuh kolom 'text' untuk diproses
# Mari kita lihat data kotornya dulu
print("--- Data Mentah ---")
print(df['text'].head())

--- Data Mentah ---
0     Sistem RAW udah di pake google pixel sejak lama😅
1            Motorola 60 pro jg SDH raw disegmen 7jtan
2    perbedaan apple log 1 dan 2 apa yaa? apakah le...
3    pake seadanya dulu aja, mau beli sayang duit. ...
4    Kebanyakan pengguna ipon cm di pake pamer  doa...
Name: text, dtype: object


In [ ]:
def cleaning_text(text):
    # 1. Case Folding (Ubah ke huruf kecil)
    text = str(text).lower()

    # 2. Hapus URL/Link (http://...)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 3. Hapus Mention (@username)
    text = re.sub(r'@[A-Za-z0-9]+', '', text)

    # 4. Hapus Hashtag (#)
    text = re.sub(r'#', '', text)

    # 5. Hapus Angka
    text = re.sub(r'\d+', '', text)

    # 6. Hapus Tanda Baca
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 7. Hapus Whitespace berlebih (spasi ganda)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Terapkan fungsi ke kolom text
df['text_clean'] = df['text'].apply(cleaning_text)

print("--- Hasil Cleaning ---")
print(df[['text', 'text_clean']].head())

--- Hasil Cleaning ---
                                                text  \
0   Sistem RAW udah di pake google pixel sejak lama😅   
1          Motorola 60 pro jg SDH raw disegmen 7jtan   
2  perbedaan apple log 1 dan 2 apa yaa? apakah le...   
3  pake seadanya dulu aja, mau beli sayang duit. ...   
4  Kebanyakan pengguna ipon cm di pake pamer  doa...   

                                          text_clean  
0   sistem raw udah di pake google pixel sejak lama😅  
1              motorola pro jg sdh raw disegmen jtan  
2  perbedaan apple log dan apa yaa apakah lebih b...  
3  pake seadanya dulu aja mau beli sayang duit ma...  
4  kebanyakan pengguna ipon cm di pake pamer doan...  


In [ ]:
# Download kamus alay dari internet (sumber populer: nasalsabila)
url_kamus_alay = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
kamus_alay = pd.read_csv(url_kamus_alay)

# Ubah jadi dictionary {kata_alay : kata_baku}
dict_alay = dict(zip(kamus_alay['slang'], kamus_alay['formal']))

# Tambahan manual (jika ada kata spesifik gadget)
additional_dict = {
    'ip': 'iphone',
    'hp': 'handphone',
    'batt': 'baterai',
    'cam': 'kamera'
}
dict_alay.update(additional_dict)

def normalisasi_alay(text):
    return ' '.join([dict_alay[word] if word in dict_alay else word for word in text.split(' ')])

# Terapkan normalisasi
df['text_normal'] = df['text_clean'].apply(normalisasi_alay)

print("--- Hasil Normalisasi ---")
print(df[['text_clean', 'text_normal']].head())

--- Hasil Normalisasi ---
                                          text_clean  \
0   sistem raw udah di pake google pixel sejak lama😅   
1              motorola pro jg sdh raw disegmen jtan   
2  perbedaan apple log dan apa yaa apakah lebih b...   
3  pake seadanya dulu aja mau beli sayang duit ma...   
4  kebanyakan pengguna ipon cm di pake pamer doan...   

                                         text_normal  
0  sistem raw sudah di pakai google pixel sejak l...  
1          motorola pro juga sudah raw disegmen jtan  
2  perbedaan apple log dan apa ya apakah lebih ba...  
3  pakai seadanya dulu saja mau beli sayang duit ...  
4  kebanyakan pengguna ipon cuma di pakai pamer d...  


In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StopWordRemoverFactory()
stopword_list = factory.get_stop_words()

# Tambahkan stopword manual jika perlu (misal kata 'video' atau 'channel' tidak penting)
stopword_list.extend(['video', 'channel', 'konten', 'nya'])

# Ubah list ke set agar pencarian lebih cepat
stopword_set = set(stopword_list)

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stopword_set])

df['text_stopword'] = df['text_normal'].apply(remove_stopwords)

print("--- Hasil Stopword Removal ---")
print(df[['text_normal', 'text_stopword']].head())

--- Hasil Stopword Removal ---
                                         text_normal  \
0  sistem raw sudah di pakai google pixel sejak l...   
1          motorola pro juga sudah raw disegmen jtan   
2  perbedaan apple log dan apa ya apakah lebih ba...   
3  pakai seadanya dulu saja mau beli sayang duit ...   
4  kebanyakan pengguna ipon cuma di pakai pamer d...   

                                       text_stopword  
0          sistem raw pakai google pixel sejak lama😅  
1                     motorola pro raw disegmen jtan  
2      perbedaan apple log apa lebih bagus apple log  
3  pakai seadanya dulu mau beli sayang duit malah...  
4  kebanyakan pengguna ipon cuma pakai pamer doan...  


In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Inisialisasi Stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Fungsi stemming
def stemming_text(text):
    return stemmer.stem(text)

print("Sedang melakukan stemming... Mohon tunggu sebentar.")
# Terapkan ke data (ini mungkin agak lama)
df['text_stemmed'] = df['text_stopword'].apply(stemming_text)

print("--- Hasil Akhir (Ready untuk Model) ---")
print(df[['text', 'text_stemmed']].head())

Sedang melakukan stemming... Mohon tunggu sebentar.
--- Hasil Akhir (Ready untuk Model) ---
                                                text  \
0   Sistem RAW udah di pake google pixel sejak lama😅   
1          Motorola 60 pro jg SDH raw disegmen 7jtan   
2  perbedaan apple log 1 dan 2 apa yaa? apakah le...   
3  pake seadanya dulu aja, mau beli sayang duit. ...   
4  Kebanyakan pengguna ipon cm di pake pamer  doa...   

                                        text_stemmed  
0           sistem raw pakai google pixel sejak lama  
1                       motorola pro raw segmen jtan  
2           beda apple log apa lebih bagus apple log  
3  pakai ada dulu mau beli sayang duit malah jatu...  
4  banyak guna ipon cuma pakai pamer doang buat t...  


In [ ]:
# Kita simpan teks aslinya (untuk referensi manusia) dan teks bersih (untuk mesin)
df_final = df[['text', 'text_stemmed']]

# Hapus baris yang kosong setelah dibersihkan (jika ada)
df_final = df_final[df_final['text_stemmed'] != '']
df_final.dropna(inplace=True)

# Simpan ke CSV
filename_bersih = "dataset_bersih_iphone17.csv"
df_final.to_csv(filename_bersih, index=False)

print(f"Sukses! Data bersih tersimpan di {filename_bersih}. Silakan download.")

Sukses! Data bersih tersimpan di dataset_bersih_iphone17.csv. Silakan download.
